In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))


In [2]:
from services.chat_service.rag_service import QdrantRetriever
from infrastructure.llm.embeddings import get_default_embeddings
from infrastructure.llm.llm_provider import get_chat_llm
from services.chat_service.rag_service import RAGService


d:\ai\bootcamp\mini-project-03\gift-concierge\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embedder = get_default_embeddings()
llm = get_chat_llm()

In [4]:
question = 'Show me birthday cakes under LKR 10000'

rag_service = RAGService(embedder=embedder, llm=llm, k=20, score_threshold=0.6)

answer = rag_service.generate(query=question)

2026-04-14 08:39:23.498 | INFO     | infrastructure.db.qdrant_client:get_qdrant_client:50 - Conntect to Qdrant cloud at https://95e8db76-61bb-4619-9da3-e9a4a2232693.us-west-1-0.aws.cloud.qdrant.io
2026-04-14 08:39:26.342 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 38 add-on chunks skipped, 0 duplicates skipped, 20 docs kept
2026-04-14 08:39:27.785 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 38 add-on chunks skipped, 0 duplicates skipped, 20 docs kept


In [5]:
print(answer)

{'answer': "🎂 **Matching Products:**\n- **Online Java Vanilla Bento Cake With C** — LKR 2840\n  Delight in the Java Vanilla Bento Cake with Cookies, a sweet treat that brings joy to every occasion.\n  🔗 [Order here](https://www.kapruka.com/buyonline/java-vanilla-bento-cake-with-c/kid/cakejava00236)\n- **Kapruka Glamorous Chocolate Box** — LKR 2920\n  Treat someone special with the Kapruka Glamorous Chocolate Box, a delightful gift for birthdays and celebrations.\n  🔗 [Order here](https://www.kapruka.com/buyonline/kapruka-glamorous-chocolate-bo/kid/chocolates00ka0095)\n- **Kapruka Coffee Brownies - 12 P** — LKR 2540\n  Treat yourself to the delicious taste of Kapruka Coffee Brownies.\n  🔗 [Order here](https://www.kapruka.com/buyonline/kapruka-coffee-brownies-12-pie/kid/chocolates00ka00123)\n- **Kapruka Cashew Brownies - 12 P** — LKR 2960\n  Discover the perfect treat with Kapruka Cashew Brownies.\n  🔗 [Order here](https://www.kapruka.com/buyonline/kapruka-cashew-brownies-12-pie/kid/choc

In [6]:
print(answer['answer'])

🎂 **Matching Products:**
- **Online Java Vanilla Bento Cake With C** — LKR 2840
  Delight in the Java Vanilla Bento Cake with Cookies, a sweet treat that brings joy to every occasion.
  🔗 [Order here](https://www.kapruka.com/buyonline/java-vanilla-bento-cake-with-c/kid/cakejava00236)
- **Kapruka Glamorous Chocolate Box** — LKR 2920
  Treat someone special with the Kapruka Glamorous Chocolate Box, a delightful gift for birthdays and celebrations.
  🔗 [Order here](https://www.kapruka.com/buyonline/kapruka-glamorous-chocolate-bo/kid/chocolates00ka0095)
- **Kapruka Coffee Brownies - 12 P** — LKR 2540
  Treat yourself to the delicious taste of Kapruka Coffee Brownies.
  🔗 [Order here](https://www.kapruka.com/buyonline/kapruka-coffee-brownies-12-pie/kid/chocolates00ka00123)
- **Kapruka Cashew Brownies - 12 P** — LKR 2960
  Discover the perfect treat with Kapruka Cashew Brownies.
  🔗 [Order here](https://www.kapruka.com/buyonline/kapruka-cashew-brownies-12-pie/kid/chocolates00ka00130)
- **Cho

---
## 🔄 CRAG Service Testing
Corrective RAG with automatic self-correction when initial retrieval confidence is low.

In [7]:
from services.chat_service.crag_service import CRAGService
from services.chat_service.rag_service import QdrantRetriever

In [8]:
# Create a shared retriever and CRAG service
retriever = QdrantRetriever(embedder=embedder, top_k=10, score_threshold=0.6)

crag_service = CRAGService(
    retriever=retriever,
    llm=llm,
    initial_k=10,
    expanded_k=20
)
print('CRAG service initialized ✅')

CRAG service initialized ✅


In [9]:
# Test CRAG generation (verbose mode shows confidence & correction steps)
crag_question = 'Show me birthday cakes under LKR 10000'

crag_result = crag_service.generate(
    query=crag_question,
    confidential_threshold=0.6,
    verbose=True
)

2026-04-14 08:39:45.755 | INFO     | services.chat_service.crag_service:generate:59 - 🔍 Query: Show me birthday cakes under LKR 10000
2026-04-14 08:39:45.756 | SUCCESS  | services.chat_service.crag_service:generate:60 - 🎯 Confidence threshold: 0.6

2026-04-14 08:39:45.757 | INFO     | services.chat_service.crag_service:generate:64 - 1️⃣  Initial retrieval (k=10)...
2026-04-14 08:39:48.726 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 38 add-on chunks skipped, 0 duplicates skipped, 10 docs kept
2026-04-14 08:39:48.728 | INFO     | services.chat_service.crag_service:generate:71 -    📊 Confidence: 0.40
2026-04-14 08:39:48.729 | WARNING  | services.chat_service.crag_service:generate:82 -    ⚠️  Low confidence - applying corrective retrieval...

2026-04-14 08:39:48.730 | INFO     | services.chat_service.crag_service:generate:86 - 2️⃣  Corrective retrieval (k=20, expanded)...
2026-04-14 08:39:49.941 | INFO     | services.chat_service.rag_servi

In [10]:
# Print the CRAG answer
print(crag_result['answer'])
print(f"\n--- CRAG Metrics ---")
print(f"Initial confidence:  {crag_result['confidence_initial']:.3f}")
print(f"Final confidence:    {crag_result['confidence_final']:.3f}")
print(f"Correction applied:  {crag_result['correction_applied']}")
print(f"Docs used:           {crag_result['docs_used']}")
print(f"Generation time:     {crag_result['generation_time']:.2f}s")
print(f"Evidence URLs:       {len(crag_result['evidence_urls'])}")

🎂 **Matching Products:**
- **Online Java Vanilla Bento Cake With C** — LKR 2840
  Delight in the Java Vanilla Bento Cake with Cookies, a sweet treat that brings joy to every occasion.
  🔗 [Order here](https://www.kapruka.com/buyonline/java-vanilla-bento-cake-with-c/kid/cakejava00236)
- **Kapruka Glamorous Chocolate Box** — LKR 2920
  Treat someone special with the Kapruka Glamorous Chocolate Box, a delightful gift for birthdays and celebrations.
  🔗 [Order here](https://www.kapruka.com/buyonline/kapruka-glamorous-chocolate-bo/kid/chocolates00ka0095)
- **Kapruka Coffee Brownies - 12 P** — LKR 2540
  Treat yourself to the delicious taste of Kapruka Coffee Brownies.
  🔗 [Order here](https://www.kapruka.com/buyonline/kapruka-coffee-brownies-12-pie/kid/chocolates00ka00123)
- **Kapruka Cashew Brownies - 12 P** — LKR 2960
  Discover the perfect treat with Kapruka Cashew Brownies.
  🔗 [Order here](https://www.kapruka.com/buyonline/kapruka-cashew-brownies-12-pie/kid/chocolates00ka00130)
- **Cho

In [11]:
# Analyze confidence for different queries (no LLM call, fast)
test_queries = [
    'Show me birthday cakes under LKR 10000',
    'Do you have chocolate gift boxes?',
    'What flowers can I send to Kandy?',
    'Do you sell cars?',
]

print(f"{'Query':<45} {'Init Conf':>10} {'Exp Conf':>10} {'Improve':>10}")
print('-' * 80)
for q in test_queries:
    analysis = crag_service.analyze_confidence(q)
    print(
        f"{q:<45} "
        f"{analysis['confidence_initial']:>10.3f} "
        f"{analysis['confidence_expanded']:>10.3f} "
        f"{analysis['improvement']:>+10.3f}"
    )

Query                                          Init Conf   Exp Conf    Improve
--------------------------------------------------------------------------------


2026-04-14 08:41:07.527 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 38 add-on chunks skipped, 0 duplicates skipped, 10 docs kept
2026-04-14 08:41:08.810 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 38 add-on chunks skipped, 0 duplicates skipped, 20 docs kept


Show me birthday cakes under LKR 10000             0.402      0.399     -0.004


2026-04-14 08:41:10.008 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 0 add-on chunks skipped, 1 duplicates skipped, 10 docs kept
2026-04-14 08:41:11.510 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 0 add-on chunks skipped, 3 duplicates skipped, 20 docs kept


Do you have chocolate gift boxes?                  0.542      0.533     -0.008
What flowers can I send to Kandy?                  0.488      0.488     +0.000


2026-04-14 08:41:14.750 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 0 add-on chunks skipped, 2 duplicates skipped, 8 docs kept
2026-04-14 08:41:15.758 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 0 add-on chunks skipped, 2 duplicates skipped, 8 docs kept


Do you sell cars?                                  0.449      0.449     +0.000


In [12]:
# Compare RAG vs CRAG on the same query
compare_query = 'Show me birthday cakes under LKR 10000'

rag_service_cmp = RAGService(embedder=embedder, llm=llm, k=10, score_threshold=0.6)
rag_result = rag_service_cmp.generate(query=compare_query)

crag_result_cmp = crag_service.generate(query=compare_query, verbose=False)

print('=' * 60)
print('RAG vs CRAG Comparison')
print('=' * 60)
print(f"\n📊 RAG:  {rag_result['num_docs']} docs, {rag_result['generation_time']:.2f}s")
print(f"📊 CRAG: {crag_result_cmp['docs_used']} docs, {crag_result_cmp['generation_time']:.2f}s")
print(f"   Correction applied: {crag_result_cmp['correction_applied']}")
print(f"   Confidence: {crag_result_cmp['confidence_initial']:.3f} → {crag_result_cmp['confidence_final']:.3f}")
print(f"\n--- RAG Answer (first 300 chars) ---")
print(rag_result['answer'][:300])
print(f"\n--- CRAG Answer (first 300 chars) ---")
print(crag_result_cmp['answer'][:300])

2026-04-14 08:41:16.983 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 38 add-on chunks skipped, 0 duplicates skipped, 10 docs kept
2026-04-14 08:41:18.266 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 38 add-on chunks skipped, 0 duplicates skipped, 10 docs kept
2026-04-14 08:41:22.405 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 38 add-on chunks skipped, 0 duplicates skipped, 10 docs kept
2026-04-14 08:41:24.120 | INFO     | services.chat_service.rag_service:_get_relevant_documents:184 - Retrieval filter: 38 add-on chunks skipped, 0 duplicates skipped, 20 docs kept


RAG vs CRAG Comparison

📊 RAG:  10 docs, 5.16s
📊 CRAG: 20 docs, 5.35s
   Correction applied: True
   Confidence: 0.402 → 0.399

--- RAG Answer (first 300 chars) ---
🎂 **Matching Products:**
  - **Online Java Vanilla Bento Cake With C** — LKR 2840
    Delight in the Java Vanilla Bento Cake with Cookies, a sweet treat that brings joy to every occasion.
    🔗 [Order here](https://www.kapruka.com/buyonline/java-vanilla-bento-cake-with-c/kid/cakejava00236)

📝 **Summ

--- CRAG Answer (first 300 chars) ---
🎂 **Matching Products:**
- **Online Java Vanilla Bento Cake With C** — LKR 2840
  Delight in the Java Vanilla Bento Cake with Cookies, a sweet treat that brings joy to every occasion.
  🔗 [Order here](https://www.kapruka.com/buyonline/java-vanilla-bento-cake-with-c/kid/cakejava00236)
- **Kapruka Gla
